In [3]:
#---------- from utility v1
import numpy as np
import pandas as pd
#import gymnasium as gym
#from gymnasium import spaces
#from stable_baselines3.common.vec_env import DummyVecEnv
#from sb3_contrib import MaskablePPO
#from sb3_contrib.common.wrappers import ActionMasker

INPUT       = "ha-2sec-full-rl-v4.pqt"
FEATS       = ["jmaD1", "haColour", "jmaD2", "rsxLast", "rsxLastD1", "rsxLastD2", "cfbD1", "haBody", "candleCross", 
               "wickAsym", "haWickTop", "haWickBott", "bodyRange", "dBody_3", "dWickTopR_3", "dWickBotR_3"]
PRICE_COL   = "Open"
LOOKBACK    = 16
COST        = 0.77
TEST_FRAC   = 0.20
N_ENVS      = 8
N_STEPS     = 2048
TOTAL_STEPS = 10_000_000
MIN_BARS    = LOOKBACK + 5
SCALE_N     = 30               # trailing bars for HA-range scale on unrealized PnL
LAMBDA      = 0.5

# ----------------------------------

df = pd.read_parquet(INPUT)
feat = df[FEATS].to_numpy(np.float32)
openp = df[PRICE_COL].to_numpy(np.float64)
dates = df["date"].values

harange = (df["haBody"].to_numpy(np.float64)
           + df["haWickTop"].to_numpy(np.float64)
           + df["haWickBott"].to_numpy(np.float64))
scale = pd.Series(harange).groupby(dates).transform(
    lambda s: s.rolling(SCALE_N, min_periods=5).mean()).to_numpy()
scale = np.where(np.isnan(scale) | (scale < 1e-6), np.nanmedian(harange), scale)

tmp = pd.DataFrame({"date": dates}); tmp["idx"] = np.arange(len(tmp))
agg = tmp.groupby("date")["idx"].agg(["first", "last"])
all_days = [d for d in agg.index if agg.loc[d, "last"] - agg.loc[d, "first"] + 1 >= MIN_BARS]
all_days = sorted(all_days)
n_test = int(len(all_days) * TEST_FRAC)
train_days = all_days[:-n_test]
test_days = all_days[-n_test:]
bounds = {d: (int(agg.loc[d, "first"]), int(agg.loc[d, "last"])) for d in all_days}
print(f"{len(all_days)} usable days -> train {len(train_days)}  test {len(test_days)}")
print(f"train {train_days[0]}..{train_days[-1]}   test {test_days[0]}..{test_days[-1]}")

tr_mask = df["date"].isin(set(train_days)).to_numpy()
mu = feat[tr_mask].mean(0); sd = feat[tr_mask].std(0) + 1e-8
feat = (feat - mu) / sd

1158 usable days -> train 927  test 231
train 2022-01-03 00:00:00..2025-08-04 00:00:00   test 2025-08-05 00:00:00..2026-06-26 00:00:00


In [4]:
"""
DP oracle -- exact utility-optimal segmentation, no RL.

3-state Markov problem per day: position in {flat, long, short}, any->any each
bar. kappa (round-trip) charged ONCE at segment birth; reversal = new segment =
new kappa. Backward induction, O(N) per day, exact.

Purpose:
  1. What does the utility ACTUALLY imply (in_pos%, marks/day, durations)?
  2. Ceiling utility/day -- the RL yardstick.
  3. (lam, kappa) calibration in seconds per combo instead of 40-min RL runs.

Run in the RL kernel (needs openp, bounds, LOOKBACK, test_days).
Pass kappa=3.14 to mirror the miscosted v3 run; kappa=1.57 is the corrected spec.
"""
import numpy as np
from tqdm import tqdm

def oracle_day(d, lam=0.5, kappa=1.57):
    s, e = bounds[d]
    i0 = s + LOOKBACK - 1
    mv = openp[i0 + 1: e + 1] - openp[i0: e]          # mv[t] at decision bar i0+t
    n = len(mv)
    shp_l = np.where(mv >= 0, mv, mv * (1 + 2 * lam))
    shp_s = np.where(-mv >= 0, -mv, -mv * (1 + 2 * lam))

    Vf = 0.0; Vl = 0.0; Vs = 0.0                       # terminal: no force-close charge
    ch = np.zeros((n, 3), np.int8)                     # argmax next-pos per (t, state)
    VF = np.empty(n); VL = np.empty(n); VS = np.empty(n)
    for t in range(n - 1, -1, -1):
        best_f = max(Vf, Vl - kappa, Vs - kappa)       # continuation from flat
        nf = int(np.argmax((Vf, Vl - kappa, Vs - kappa)))
        best_l = max(Vf, Vl, Vs - kappa)               # from long: exit free, reverse pays
        nl = int(np.argmax((Vf, Vl, Vs - kappa)))
        best_s = max(Vf, Vl - kappa, Vs)
        ns = int(np.argmax((Vf, Vl - kappa, Vs)))
        VF[t] = best_f
        VL[t] = shp_l[t] + best_l
        VS[t] = shp_s[t] + best_s
        ch[t, 0] = nf; ch[t, 1] = nl; ch[t, 2] = ns
        Vf, Vl, Vs = VF[t], VL[t], VS[t]

    acts = np.full(e - s + 1, -1, np.int8)             # per-bar action 0/1/2
    p = 0; marks = 0; in_pos = 0; durs = []; cur = 0
    for t in range(n):
        nxt = ch[t, p]
        if p != 0:
            in_pos += 1; cur += 1
        acts[i0 + t - s] = p if p != 2 else 2          # record CURRENT position as action
        if nxt != p:
            if p != 0:
                durs.append(cur); cur = 0
            if nxt != 0:
                marks += 1
        p = nxt
    if cur:
        durs.append(cur)
    durs = np.array(durs, np.float64) if durs else np.array([0.0])
    return dict(util=VF[0], marks=marks, in_pos=in_pos / n,
                dur_med=float(np.median(durs)), frac_short=float((durs <= 2).mean()),
                acts=acts)

def oracle_stats(days, lam=0.5, kappa=1.57, label=""):
    rs = [oracle_day(d, lam, kappa) for d in tqdm(days, desc=f"oracle {label}", leave=False)]
    u  = np.mean([r["util"] for r in rs]);   m  = np.mean([r["marks"] for r in rs])
    ip = np.mean([r["in_pos"] for r in rs]); dm = np.median([r["dur_med"] for r in rs])
    fs = np.mean([r["frac_short"] for r in rs])
    print(f"lam={lam:4.2f} kappa={kappa:5.2f}  util/day {u:9.1f}  marks/day {m:6.1f}  "
          f"in_pos {ip:5.1%}  dur_med {dm:5.1f}  frac<=2 {fs:5.1%}")

def to_acts5(acts3):
    """Convert per-bar position actions {0,1,2} to plot_day's 5-code scheme."""
    a5 = np.full_like(acts3, -1)
    p = 0
    for k, a in enumerate(acts3):
        if a == -1: continue
        pos = 0 if a == 0 else (1 if a == 1 else -1)
        if pos == p:
            a5[k] = 0 if pos == 0 else 3               # stay-out / hold
        elif pos == 0:
            a5[k] = 4                                   # exit
        else:
            a5[k] = 1 if pos == 1 else 2                # (re)entry incl. reversal
        p = pos
    return a5

# 1) corrected spec vs the miscosted v3 toll -- full test set
oracle_stats(test_days, lam=0.5, kappa=1.57, label="corrected")
oracle_stats(test_days, lam=0.5, kappa=3.14, label="v3-toll")

# 2) calibration sweep, subsampled days -- tune to business expectation, then
#    render a few days: plot_day-style with to_acts5(oracle_day(d, lam, kappa)["acts"])
sub = test_days[::5]
for lam in (0.1, 0.25, 0.5, 1.0):
    for kappa in (0.4, 0.8, 1.57, 3.0):
        oracle_stats(sub, lam, kappa, label="sweep")

lam=0.50 kappa= 1.57  util/day    5882.1  marks/day 1112.1  in_pos 84.4%  dur_med   7.0  frac<=2  4.3%


lam=0.50 kappa= 3.14  util/day    4520.5  marks/day  683.7  in_pos 72.8%  dur_med  11.0  frac<=2  1.0%


lam=0.10 kappa= 0.40  util/day    7732.7  marks/day 1891.2  in_pos 98.3%  dur_med   5.0  frac<=2 19.3%


lam=0.10 kappa= 0.80  util/day    7078.4  marks/day 1435.0  in_pos 97.2%  dur_med   6.0  frac<=2 10.2%


lam=0.10 kappa= 1.57  util/day    6157.2  marks/day 1005.9  in_pos 95.1%  dur_med   9.0  frac<=2  3.8%


...

lam=0.50 kappa= 3.00  util/day    4702.7  marks/day  716.1  in_pos 74.2%  dur_med  11.0  frac<=2  1.2%


lam=1.00 kappa= 0.40  util/day    7672.2  marks/day 2097.6  in_pos 93.2%  dur_med   4.0  frac<=2 22.1%


lam=1.00 kappa= 0.80  util/day    6933.7  marks/day 1641.8  in_pos 87.9%  dur_med   5.0  frac<=2 12.1%


lam=1.00 kappa= 1.57  util/day    5868.1  marks/day 1174.1  in_pos 79.3%  dur_med   7.0  frac<=2  4.7%


lam=1.00 kappa= 3.00  util/day    4529.1  marks/day  753.4  in_pos 66.2%  dur_med   9.0  frac<=2  1.3%


In [5]:
"""
plot_day("2025-03-14") -- SC-style visual validation of the RL segmentation.

Replays the saved MaskablePPO policy over one day and renders:
  - HA candlesticks (Open/High/Low/Close)
  - jma line
  - the policy's 5 actions as distinct markers:
      long entry   green triangle-up      (below bar)
      short entry  red   triangle-down    (above bar)
      exit         black x                (at close)
      hold         faint blue dot         (at close, small)
      stay-out     faint grey dot         (at low, small)

Assumes in the notebook there is already:
  df        -- full parquet incl. Open/High/Low/Close, jma, date, and the 16 FEATS
  feat      -- normalized feature matrix used in training (same normalization!)
  bounds    -- {date: (first_idx, last_idx)}
  TradingEnv, FEATS, LOOKBACK, ACT_* constants from the env script
  model     -- loaded via: model = MaskablePPO.load("rl_seg_v1")

If replaying in a fresh kernel, rebuild feat/bounds with the SAME train-day
normalization as training, or the policy sees shifted inputs and marks garbage.
"""
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ACTION_STYLE = {
    0: dict(name="stay-out", symbol="circle",        color="rgba(32,32,32,1)", size=5),
    1: dict(name="long",     symbol="triangle-up",   color="#00b300",                size=11),
    2: dict(name="short",    symbol="triangle-down", color="#e60000",                size=11),
    3: dict(name="hold",     symbol="circle",        color="rgba(30,100,220,0.6)",  size=5),
    4: dict(name="exit",     symbol="x",             color="#000000",                size=6),
}

def plot_day(date, jma_col="jma", show_holds=True, height=750):
    d = pd.Timestamp(date)
    if d not in bounds:
        avail = sorted(bounds.keys())
        raise ValueError(f"{date} not in data. range {avail[0]}..{avail[-1]}")
    s, e = bounds[d]

    # ---- replay policy over the day, greedy, masked ----
    env = TradingEnv([d])
    obs, _ = env.reset(options={"day": d})
    acts = np.full(e - s + 1, -1, np.int8)         # per-bar action, indexed from s
    done = False
    while not done:
        i_rel = env.i - s                          # decision bar (relative)
        mask = env.action_masks()
        a, _ = model.predict(obs, action_masks=mask, deterministic=True)
        acts[i_rel] = int(a)
        obs, r, done, _, info = env.step(int(a))

    day = df.iloc[s:e + 1]
    ts = pd.to_datetime(day["timestamp"])

    fig = make_subplots(rows=1, cols=1)
    fig.add_trace(go.Candlestick(
        x=ts, open=day["Open"], high=day["High"], low=day["Low"], close=day["Close"],
        name="HA", increasing_line_color="#26a69a", decreasing_line_color="#ef5350",
        line=dict(width=1), opacity=0.9))
    if jma_col in day.columns:
        fig.add_trace(go.Scatter(x=ts, y=day[jma_col], name="jma",
                                 line=dict(color="#0000ff", width=1.6)))
    else:
        print(f"[warn] '{jma_col}' not in df; candidates: "
              f"{[c for c in day.columns if 'jma' in c.lower()]}")

    lo, hi = day["Low"].values, day["High"].values
    pad = (hi - lo).mean() * 0.6
    for code, st in ACTION_STYLE.items():
        if not show_holds and code in (0, 3):
            continue
        m = acts == code
        if not m.any():
            continue
        if code == 1:   y = lo[m] - pad            # long entry below bar
        elif code == 2: y = hi[m] + pad            # short entry above bar
        elif code == 4: y = day["Close"].values[m] # exit at close
        elif code == 3: y = day["Close"].values[m] # hold at close
        else:           y = lo[m] - pad * 0.5      # stay-out at low
        fig.add_trace(go.Scatter(
            x=ts[m], y=y, mode="markers", name=st["name"],
            marker=dict(symbol=st["symbol"], color=st["color"], size=st["size"])))

    n_long = int((acts == 1).sum()); n_short = int((acts == 2).sum())
    n_exit = int((acts == 4).sum()); pct_in = float(np.isin(acts, (1, 2, 3)).mean())
    fig.update_layout(
        title=f"{pd.Timestamp(date).date()}   longs {n_long}  shorts {n_short}  "
              f"exits {n_exit}  in-segment {pct_in:.0%}",
        height=height, xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", y=1.02, x=0),
        margin=dict(l=40, r=20, t=60, b=30))
    fig.show()
    return acts                                     # per-bar actions if you want to inspect

In [6]:
TEST_DAY = "2025-01-16"
acts = plot_day(TEST_DAY)
print(acts)

NameError: name 'TradingEnv' is not defined

In [ ]:
import matplotlib.pyplot as plt
import torch

@torch.no_grad()
def action_probs_day(day):
    env = TradingEnv([day])
    obs, _ = env.reset(options={"day": day})
    out = []; done = False
    while not done:
        mask = env.action_masks()
        o = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
        try:
            dist = model.policy.get_distribution(o, action_masks=torch.as_tensor(mask).unsqueeze(0))
            p = dist.distribution.probs.squeeze(0).numpy()
        except TypeError:
            dist = model.policy.get_distribution(o)
            p = dist.distribution.probs.squeeze(0).numpy()
            p = p * mask; p = p / p.sum()
        out.append(p)
        obs, r, done, _, _ = env.step(int(np.argmax(p)))
    return np.array(out)

P = action_probs_day(pd.Timestamp(TEST_DAY))   # the screenshot day
plt.figure(figsize=(18,4))
plt.plot(P[:,0], label="stay-out", lw=.8)
plt.plot(P[:,1], label="long", lw=.8)
plt.plot(P[:,2], label="short", lw=.8)
plt.legend(); plt.show()